In [1]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

In [2]:
ratings = pd.read_csv(
    "../data/raw/ratings.dat",
    sep="::",
    engine="python",
    names=["user_id", "movie_id", "rating", "timestamp"],
    encoding="latin-1"
)

users = pd.read_csv(
    "../data/raw/users.dat",
    sep="::",
    engine="python",
    names=["user_id", "gender", "age", "occupation", "zip_code"],
    encoding="latin-1"
)

movies = pd.read_csv(
    "../data/raw/movies.dat",
    sep="::",
    engine="python",
    names=["movie_id", "title", "genres"],
    encoding="latin-1"
)

In [3]:
train_data = pd.read_csv("../data/processed/train.csv")
test_data = pd.read_csv("../data/processed/test.csv")

In [4]:
positive_train = train_data[train_data["rating"] >= 4]

print("Số lượt đánh giá tích cực:", len(positive_train))
print("Số user có đánh giá tích cực:", positive_train["user_id"].nunique())
print("Số phim có đánh giá tích cực:", positive_train["movie_id"].nunique())

Số lượt đánh giá tích cực: 460119
Số user có đánh giá tích cực: 6037
Số phim có đánh giá tích cực: 3499


In [5]:
movie_popularity = (
    train_data[train_data["rating"] >= 4]
    .groupby("movie_id")
    .size()
    .sort_values(ascending=False)
)

movie_popularity.head(10)

movie_id
2858    2266
260     2071
1196    2028
593     1815
2028    1809
1198    1790
2762    1719
2571    1712
1210    1697
527     1664
dtype: int64

In [6]:
top_10_popular = (
    movie_popularity.rename("positive_count")
    .reset_index()
    .merge(movies[["movie_id", "title", "genres"]], on="movie_id", how="left")
)

top_10_popular.head(10)

,movie_id,positive_count,title,genres
0,2858,2266,American Beauty (1999),Comedy|Drama
1,260,2071,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Fantasy|Sci-Fi
2,1196,2028,Star Wars: Episode V - The Empire Strikes Back...,Action|Adventure|Drama|Sci-Fi|War
3,593,1815,"Silence of the Lambs, The (1991)",Drama|Thriller
4,2028,1809,Saving Private Ryan (1998),Action|Drama|War
5,1198,1790,Raiders of the Lost Ark (1981),Action|Adventure
6,2762,1719,"Sixth Sense, The (1999)",Thriller
7,2571,1712,"Matrix, The (1999)",Action|Sci-Fi|Thriller
8,1210,1697,Star Wars: Episode VI - Return of the Jedi (1983),Action|Adventure|Romance|Sci-Fi|War
9,527,1664,Schindler's List (1993),Drama|War


In [7]:
user_seen_movies = train_data.groupby("user_id")["movie_id"].apply(set)

In [8]:
def recommend_popular_movies(user_id, top_n=10):
    # Lấy danh sách phim người dùng đã đánh giá trong tập train
    seen_movies = user_seen_movies.get(user_id, set())

    # Lọc bỏ các phim đã được đánh giá
    recommendations = movie_popularity[
        ~movie_popularity.index.isin(seen_movies)
    ]

    # Lấy Top-N phim phổ biến nhất còn lại
    return recommendations.head(top_n).index.tolist()

In [9]:
recommended_ids = recommend_popular_movies(user_id=1, top_n=10)

recommended_movies = movies[
    movies["movie_id"].isin(recommended_ids)
][["movie_id", "title", "genres"]]

recommended_movies

,movie_id,title,genres
315,318,"Shawshank Redemption, The (1994)",Drama
585,589,Terminator 2: Judgment Day (1991),Action|Sci-Fi|Thriller
589,593,"Silence of the Lambs, The (1991)",Drama|Thriller
604,608,Fargo (1996),Crime|Drama|Thriller
847,858,"Godfather, The (1972)",Action|Crime|Drama
1178,1196,Star Wars: Episode V - The Empire Strikes Back...,Action|Adventure|Drama|Sci-Fi|War
1180,1198,Raiders of the Lost Ark (1981),Action|Adventure
1192,1210,Star Wars: Episode VI - Return of the Jedi (1983),Action|Adventure|Romance|Sci-Fi|War
2502,2571,"Matrix, The (1999)",Action|Sci-Fi|Thriller
2789,2858,American Beauty (1999),Comedy|Drama


In [10]:
recommended_ids = recommend_popular_movies(user_id=1, top_n=10)

recommended_movies = (
    pd.DataFrame({"movie_id": recommended_ids})
    .merge(movies[["movie_id", "title", "genres"]], on="movie_id", how="left")
    .merge(
        movie_popularity.rename("positive_count"),
        left_on="movie_id",
        right_index=True,
        how="left"
    )
)

recommended_movies

,movie_id,title,genres,positive_count
0,2858,American Beauty (1999),Comedy|Drama,2266
1,1196,Star Wars: Episode V - The Empire Strikes Back...,Action|Adventure|Drama|Sci-Fi|War,2028
2,593,"Silence of the Lambs, The (1991)",Drama|Thriller,1815
3,1198,Raiders of the Lost Ark (1981),Action|Adventure,1790
4,2571,"Matrix, The (1999)",Action|Sci-Fi|Thriller,1712
5,1210,Star Wars: Episode VI - Return of the Jedi (1983),Action|Adventure|Romance|Sci-Fi|War,1697
6,608,Fargo (1996),Crime|Drama|Thriller,1658
7,318,"Shawshank Redemption, The (1994)",Drama,1649
8,589,Terminator 2: Judgment Day (1991),Action|Sci-Fi|Thriller,1649
9,858,"Godfather, The (1972)",Action|Crime|Drama,1577


In [11]:
# Các phim được đánh giá tích cực trong tập test
test_relevant = (
    test_data[test_data["rating"] >= 4]
    .groupby("user_id")["movie_id"]
    .apply(set)
)

# Chỉ đánh giá các phim đã xuất hiện trong tập train
train_movie_ids = set(train_data["movie_id"].unique())

test_relevant = test_relevant.apply(
    lambda movie_ids: movie_ids & train_movie_ids
)

# Loại bỏ các user không còn phim liên quan sau khi lọc
test_relevant = test_relevant[
    test_relevant.apply(len) > 0
]

print("Số user có phim liên quan trong tập test:", len(test_relevant))
print(
    "Tổng số phim liên quan:",
    sum(len(movie_ids) for movie_ids in test_relevant)
)

Số user có phim liên quan trong tập test: 5994
Tổng số phim liên quan: 115155


In [12]:
k = 10
results = []

# Danh sách phim được xếp hạng theo độ phổ biến từ train
ranked_movie_ids = movie_popularity.index.tolist()

for user_id, relevant_movies in test_relevant.items():
    seen_movies = user_seen_movies.get(user_id, set())

    # Lấy Top-10 phim phổ biến mà người dùng chưa đánh giá trong train
    recommendations = []
    for movie_id in ranked_movie_ids:
        if movie_id not in seen_movies:
            recommendations.append(movie_id)

        if len(recommendations) == k:
            break

    # Xác định phim được đề xuất có nằm trong tập liên quan không
    hits = [
        1 if movie_id in relevant_movies else 0
        for movie_id in recommendations
    ]

    hit_count = sum(hits)

    precision = hit_count / k
    recall = hit_count / len(relevant_movies)

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0
        else 0
    )

    # Tính DCG@10 và IDCG@10 để tính NDCG@10
    dcg = sum(
        hit / np.log2(rank + 2)
        for rank, hit in enumerate(hits)
    )

    ideal_hits = min(len(relevant_movies), k)
    idcg = sum(
        1 / np.log2(rank + 2)
        for rank in range(ideal_hits)
    )

    ndcg = dcg / idcg if idcg > 0 else 0

    results.append({
        "user_id": user_id,
        "Precision@10": precision,
        "Recall@10": recall,
        "F1@10": f1,
        "NDCG@10": ndcg
    })

# Tổng hợp kết quả trung bình trên các user
baseline_results = pd.DataFrame(results)

print("Số user được đánh giá:", len(baseline_results))
print("\nKết quả Popularity-Based Baseline:")
print(
    baseline_results[
        ["Precision@10", "Recall@10", "F1@10", "NDCG@10"]
    ].mean().round(4)
)

Số user được đánh giá: 5994

Kết quả Popularity-Based Baseline:
Precision@10    0.1519
Recall@10       0.0907
F1@10           0.0918
NDCG@10         0.1766
dtype: float64
